# DBSQL JSON & VARIANT Demo

A hands-on walkthrough of querying semi-structured JSON data in Databricks SQL,
using synthetic healthcare payer data (claims, members, providers, operational events).

**Run `00_setup.ipynb` first** to create the tables before executing any cells here.

---

## ⚠️  Set your catalog name before running

Enter the same catalog name you used in the setup notebook.


In [ ]:
dbutils.widgets.text("catalog_name", "", "⚠️ Enter your catalog name (same as setup)")
catalog_name = dbutils.widgets.get("catalog_name")

if not catalog_name.strip():
    raise ValueError(
        "\n\n❌  catalog_name widget is empty.\n"
        "    Enter your catalog name in the widget at the top of the notebook,\n"
        "    then re-run this cell before continuing.\n"
    )
print(f"✅  Catalog: {catalog_name}")


In [ ]:
%sql
USE CATALOG ${catalog_name};

## The dataset

| Schema | Table | Rows | JSON column | Type |
|--------|-------|------|-------------|------|
| `claims_json_demo` | `claims_submissions` | 500 | `claim_json` | STRING |
| `member_json_demo` | `member_profiles` | 300 | `profile_json` | STRING |
| `provider_json_demo` | `provider_network` | 200 | `provider_data` | **VARIANT** |
| `ops_json_demo` | `operational_events` | 400 | `event_json` | STRING |

Each table has structured columns alongside a JSON column — exactly like data that
arrives from REST APIs, FHIR feeds, and operational systems in the real world.


In [ ]:
%sql
-- Quick sanity check — confirm all four tables are loaded
SELECT 'claims'    AS table_name, count(*) AS rows FROM claims_json_demo.claims_submissions
UNION ALL SELECT 'members',   count(*) FROM member_json_demo.member_profiles
UNION ALL SELECT 'providers', count(*) FROM provider_json_demo.provider_network
UNION ALL SELECT 'ops_events',count(*) FROM ops_json_demo.operational_events
ORDER BY table_name;


---
## Part 1 — JSON String Basics: Extraction Fundamentals

The `:` (colon) operator is the primary way to extract fields from a JSON string column.
Think of it as "inside this column, give me this field."


### Scenario 1 — Extract top-level fields with colon syntax

The colon operator returns values as STRING by default.


In [ ]:
%sql
SELECT
  claim_id,
  claim_json:claim_number              AS claim_number,
  claim_json:filing_code               AS filing_code,
  claim_json:release_of_info           AS release_of_info,
  claim_json:coordination_of_benefits  AS cob_flag
FROM claims_json_demo.claims_submissions
LIMIT 10;


### Scenario 2 — Navigate nested objects with dot notation

Use dots to drill into nested JSON objects — subscriber info, billing provider details, etc.


In [ ]:
%sql
SELECT
  claim_id,
  claim_json:subscriber.member_id          AS member_id,
  claim_json:subscriber.group_number       AS group_number,
  claim_json:subscriber.relationship_code  AS relationship,
  claim_json:billing_provider.npi          AS billing_npi,
  claim_json:billing_provider.name         AS billing_provider_name
FROM claims_json_demo.claims_submissions
LIMIT 10;


### Scenario 3 — Case sensitivity: colon vs bracket paths

Colon paths are **case-insensitive**.  Bracket paths `['field']` are **case-sensitive**.
This matters when working with data from external systems where field-name casing may vary.


In [ ]:
%sql
-- All three return the same value
SELECT
  claim_id,
  claim_json:subscriber.member_id   AS lowercase_path,
  claim_json:SUBSCRIBER.MEMBER_ID   AS uppercase_path,
  claim_json:Subscriber.Member_Id   AS mixed_path
FROM claims_json_demo.claims_submissions LIMIT 3;


In [ ]:
%sql
-- Brackets are case-sensitive — wrong case returns NULL
SELECT
  claim_id,
  claim_json:['subscriber']   AS correct_case,
  claim_json:['SUBSCRIBER']   AS wrong_case_returns_null
FROM claims_json_demo.claims_submissions LIMIT 3;


### Scenario 4 — Cast extracted values to proper types

By default, colon extraction returns STRING.  Use `::type` to cast for math and comparisons.
This is the most common mistake new users make — forgetting to cast before doing arithmetic.


In [ ]:
%sql
-- Without casting — values come back as quoted strings
SELECT
  claim_id,
  claim_json:adjudication.paid_amount                    AS paid_raw_string,
  typeof(claim_json:adjudication.paid_amount)            AS what_type_is_this
FROM claims_json_demo.claims_submissions LIMIT 5;


In [ ]:
%sql
-- With casting — now we can do math and proper filtering
SELECT
  claim_id,
  claim_json:adjudication.paid_amount::double        AS paid_amount,
  claim_json:adjudication.allowed_amount::double      AS allowed_amount,
  claim_json:coordination_of_benefits::boolean        AS has_cob,
  round(claim_json:adjudication.paid_amount::double /
    NULLIF(claim_json:adjudication.allowed_amount::double, 0) * 100, 1) AS pct_of_allowed
FROM claims_json_demo.claims_submissions
WHERE claim_json:adjudication.status::string = 'paid'
  AND claim_json:adjudication.paid_amount::double > 500
ORDER BY paid_amount DESC
LIMIT 10;


In [ ]:
%sql
-- Available casts: ::string, ::double, ::int, ::boolean, ::date, ::timestamp
-- Operational events — extract SLA fields
SELECT
  event_id,
  event_json:status::string          AS event_status,
  event_json:priority::string        AS priority,
  event_json:assigned_to::string     AS assigned_to,
  event_json:sla.target_hours::int   AS sla_target_hrs,
  event_json:sla.actual_hours::double AS sla_actual_hrs,
  event_json:sla.met::boolean        AS sla_met
FROM ops_json_demo.operational_events
LIMIT 10;


---
## Part 2 — Working with Arrays

In traditional databases, a claim's service lines would live in a child table.
In JSON they are embedded as an array.  Here is how to work with them.


### Scenario 5 — Access array elements by index

Arrays are zero-based.  Use `[0]`, `[1]`, etc.  If the index does not exist, you get NULL — no error.


In [ ]:
%sql
SELECT
  claim_id,
  claim_json:service_lines[0].procedure_code        AS first_line_cpt,
  claim_json:service_lines[0].charge_amount::double  AS first_line_charge,
  claim_json:service_lines[0].place_of_service      AS first_line_pos,
  claim_json:service_lines[1].procedure_code        AS second_line_cpt,
  claim_json:service_lines[1].charge_amount::double  AS second_line_charge
FROM claims_json_demo.claims_submissions
LIMIT 10;


### Scenario 6 — Wildcard extraction with `[*]`

`[*]` pulls a field from **every** element in the array and returns a JSON array of those values.
Great for quick lookups without needing to explode the whole array.


In [ ]:
%sql
SELECT
  claim_id,
  claim_json:service_lines[*].procedure_code   AS all_procedure_codes,
  claim_json:service_lines[*].charge_amount     AS all_charges
FROM claims_json_demo.claims_submissions
LIMIT 10;


In [ ]:
%sql
-- Use wildcard + array_contains to find claims that include a specific CPT code
SELECT claim_id, claim_type, total_charge_amount,
  claim_json:service_lines[*].procedure_code AS all_cpts
FROM claims_json_demo.claims_submissions
WHERE array_contains(
  from_json(claim_json:service_lines[*].procedure_code, 'ARRAY<STRING>'),
  '99214'
);


### Scenario 7 — EXPLODE service lines into one row per line

This is the most powerful technique.  `from_json` + `explode` "un-nests" an array into
individual rows — equivalent to unnesting a child table.

**Step 1: discover the schema first.**


In [ ]:
%sql
-- Step 1: Let Databricks infer the schema — copy the output for Step 2
SELECT schema_of_json(claim_json:service_lines) AS inferred_schema
FROM claims_json_demo.claims_submissions LIMIT 1;


In [ ]:
%sql
-- Step 2: Explode into individual rows
SELECT
  c.claim_id, c.claim_type, c.submitted_date,
  sl.line_number, sl.procedure_code, sl.modifiers,
  sl.charge_amount, sl.place_of_service, sl.rendering_provider_npi
FROM claims_json_demo.claims_submissions c
LATERAL VIEW OUTER explode(
  from_json(c.claim_json:service_lines,
    'ARRAY<STRUCT<line_number:INT, procedure_code:STRING, modifiers:ARRAY<STRING>,
     diagnosis_pointers:ARRAY<STRING>, units:INT, charge_amount:DOUBLE,
     place_of_service:STRING, date_of_service:STRING,
     rendering_provider_npi:STRING, revenue_code:STRING>>')
) sl_tbl AS sl
LIMIT 20;


In [ ]:
%sql
-- Step 3: Aggregate at the line level — top CPT codes by volume and charge
-- (Only declare the fields you actually need in the schema string)
SELECT
  sl.procedure_code,
  count(*)                        AS line_count,
  round(sum(sl.charge_amount), 2) AS total_charges,
  round(avg(sl.charge_amount), 2) AS avg_charge
FROM claims_json_demo.claims_submissions c
LATERAL VIEW explode(
  from_json(c.claim_json:service_lines,
    'ARRAY<STRUCT<procedure_code:STRING, charge_amount:DOUBLE>>')
) sl_tbl AS sl
GROUP BY sl.procedure_code
ORDER BY line_count DESC;


### Scenario 8 — Explode member engagement history

Same pattern works on deeply nested arrays.  This is how you build outreach
effectiveness reports from member engagement data.


In [ ]:
%sql
-- Flatten engagement history — one row per outreach touch
SELECT
  m.member_id, m.line_of_business,
  m.profile_json:demographics.first_name::string AS first_name,
  m.profile_json:demographics.last_name::string  AS last_name,
  eng.event_type, eng.channel, eng.timestamp, eng.outcome, eng.agent_id
FROM member_json_demo.member_profiles m
LATERAL VIEW OUTER explode(
  from_json(m.profile_json:engagement_history,
    'ARRAY<STRUCT<event_type:STRING, channel:STRING, timestamp:STRING,
     outcome:STRING, agent_id:STRING>>')
) eng_tbl AS eng
WHERE eng.event_type IS NOT NULL
LIMIT 20;


In [ ]:
%sql
-- Aggregate: outreach effectiveness by channel and outcome
SELECT
  eng.channel, eng.outcome,
  count(*) AS touch_count,
  count(DISTINCT m.member_id) AS unique_members
FROM member_json_demo.member_profiles m
LATERAL VIEW explode(
  from_json(m.profile_json:engagement_history,
    'ARRAY<STRUCT<event_type:STRING, channel:STRING, timestamp:STRING,
     outcome:STRING, agent_id:STRING>>')
) eng_tbl AS eng
WHERE eng.event_type IS NOT NULL
GROUP BY eng.channel, eng.outcome
ORDER BY eng.channel, touch_count DESC;


### Scenario 8B — Member conditions array

Same explode pattern, applied to the active conditions array.


In [ ]:
%sql
-- Most common active conditions in the Medicare Advantage population
SELECT
  cond.code AS dx_code,
  cond.description,
  count(DISTINCT m.member_id) AS member_count,
  round(count(DISTINCT m.member_id) * 100.0 /
    (SELECT count(*) FROM member_json_demo.member_profiles WHERE line_of_business = 'Medicare Advantage'), 1
  ) AS pct_of_ma_members
FROM member_json_demo.member_profiles m
LATERAL VIEW explode(
  from_json(m.profile_json:conditions,
    'ARRAY<STRUCT<code:STRING, description:STRING, onset_date:STRING, status:STRING>>')
) cond_tbl AS cond
WHERE m.line_of_business = 'Medicare Advantage'
  AND cond.status = 'active'
GROUP BY cond.code, cond.description
ORDER BY member_count DESC;


---
## Part 3 — JSON Functions Toolkit

Beyond colon syntax, Databricks SQL has a set of functions for working with JSON strings.


### Scenario 9 — `from_json`: parse into a struct

When you need many fields from the same nested object, parse it once into a struct.
Then use normal `.field` syntax — no quoting, no repeated parsing.


In [ ]:
%sql
-- Parse adjudication into a struct — one parse, clean field access
SELECT claim_id, adj.status, adj.paid_amount, adj.allowed_amount,
  adj.denial_codes, adj.remark_codes
FROM (
  SELECT claim_id,
    from_json(claim_json:adjudication,
      'STRUCT<status:STRING, paid_amount:DOUBLE, allowed_amount:DOUBLE,
       denial_codes:ARRAY<STRING>, remark_codes:ARRAY<STRING>>') AS adj
  FROM claims_json_demo.claims_submissions
)
WHERE adj.status = 'denied'
LIMIT 10;


### Scenario 10 — `get_json_object`: JSONPath-style extraction

If you are coming from Oracle or another database, `get_json_object` uses `$.field`
syntax that may feel familiar.  Functionally equivalent to colon syntax.


In [ ]:
%sql
-- Colon syntax vs get_json_object — same results
SELECT claim_id,
  claim_json:subscriber.member_id                       AS colon_member_id,
  get_json_object(claim_json, '$.subscriber.member_id') AS gjso_member_id,
  claim_json:service_lines[0].procedure_code             AS colon_first_cpt,
  get_json_object(claim_json, '$.service_lines[0].procedure_code') AS gjso_first_cpt
FROM claims_json_demo.claims_submissions LIMIT 5;


### Scenario 11 — `json_tuple`: extract multiple top-level fields at once

Efficient when you need several top-level fields — avoids re-parsing the JSON for each one.


In [ ]:
%sql
SELECT e.event_id, e.event_type, jt.status, jt.priority, jt.assigned_to
FROM ops_json_demo.operational_events e
LATERAL VIEW json_tuple(e.event_json, 'status', 'priority', 'assigned_to')
  jt AS status, priority, assigned_to
LIMIT 10;


### Scenario 12 — `schema_of_json`: inspect unknown JSON structure

When you receive a new data feed and do not know the JSON shape,
`schema_of_json` infers the schema for you.  The output is copy-paste ready for `from_json`.


In [ ]:
%sql
SELECT schema_of_json(claim_json) AS claim_schema
FROM claims_json_demo.claims_submissions LIMIT 1;


In [ ]:
%sql
SELECT schema_of_json(profile_json) AS profile_schema
FROM member_json_demo.member_profiles LIMIT 1;


In [ ]:
%sql
SELECT schema_of_json(event_json) AS event_schema
FROM ops_json_demo.operational_events WHERE event_type = 'prior_auth' LIMIT 1;


### Scenario 13 — NULL behavior in JSON

Important subtlety: a **missing key** and an **explicit JSON null** both become SQL NULL.
An **empty string** is not null.  Understanding this prevents subtle analytic bugs.


In [ ]:
%sql
SELECT
  get_json_object('{"key": null}', '$.key') IS NULL   AS json_null_is_sql_null,
  get_json_object('{"other": 1}',  '$.key') IS NULL   AS missing_key_is_sql_null,
  get_json_object('{"key": ""}',   '$.key')            AS empty_string_value,
  get_json_object('{"key": ""}',   '$.key') IS NULL   AS empty_string_is_not_null;


In [ ]:
%sql
-- Practical example: resolution is null for unresolved events
SELECT event_id,
  event_json:status::string             AS status,
  event_json:resolution IS NULL         AS resolution_is_null,
  event_json:resolution.outcome::string AS outcome
FROM ops_json_demo.operational_events
LIMIT 10;


In [ ]:
%sql
-- Count resolved vs unresolved
SELECT
  CASE WHEN event_json:resolution IS NULL OR event_json:resolution::string = 'null'
       THEN 'Unresolved' ELSE 'Resolved' END AS resolution_status,
  count(*) AS event_count
FROM ops_json_demo.operational_events
GROUP BY 1;


---
## Part 4 — The VARIANT Type

Everything above uses STRING columns with JSON text.  The **VARIANT** type is purpose-built
for semi-structured data.

| | STRING JSON | VARIANT |
|--|--|--|
| Parsing | Re-parsed on **every query** | Parsed **once** at write time |
| Validation | Errors at query time | Bad JSON caught at ingest |
| Type handling | Everything returns as string | Returns native types |
| Performance | Re-parse overhead at scale | Optimized binary format |

The `provider_network` table uses VARIANT — all scenarios in this section use it.


### Scenario 14 — VARIANT vs STRING: same syntax, different internals


In [ ]:
%sql
-- See that provider_data is VARIANT, not STRING
SELECT provider_id, provider_type, typeof(provider_data) AS data_type
FROM provider_json_demo.provider_network LIMIT 3;


In [ ]:
%sql
-- Same colon syntax works on VARIANT — but no re-parsing underneath
SELECT p.provider_id,
  p.provider_data:name                    AS name_obj,
  p.provider_data:contract.contract_type  AS contract_type,
  p.provider_data:contract.discount_pct   AS discount_pct
FROM provider_json_demo.provider_network p LIMIT 5;


### Scenario 15 — `PARSE_JSON` and `TRY_PARSE_JSON`

This is how data gets into a VARIANT column.
`TRY_PARSE_JSON` returns NULL for malformed JSON instead of raising an error —
use it in production pipelines.


In [ ]:
%sql
SELECT PARSE_JSON('{"name": "Dr. Smith", "npi": "1234567890"}') AS parsed;


In [ ]:
%sql
SELECT
  TRY_PARSE_JSON('{"valid": true}') AS good_json,
  TRY_PARSE_JSON('{ bad json }')    AS bad_json_returns_null;


In [ ]:
%sql
DESCRIBE TABLE provider_json_demo.provider_network;

### Scenario 16 — Colon syntax on VARIANT: typed values

When extracting from VARIANT, numbers come back as numbers — not as string representations.
You still cast for downstream aggregations, but the underlying type is already correct.


In [ ]:
%sql
SELECT
  provider_id, provider_type,
  provider_data:name.first::string         AS first_name,
  provider_data:name.last::string          AS last_name,
  provider_data:name.credentials::string   AS credentials,
  provider_data:name.organization_name::string AS org_name
FROM provider_json_demo.provider_network LIMIT 10;


In [ ]:
%sql
-- VARIANT returns typed values — discount_pct is already numeric
SELECT provider_id,
  provider_data:contract.discount_pct          AS discount_raw,
  typeof(provider_data:contract.discount_pct)  AS value_type,
  provider_data:contract.discount_pct::double  AS discount_typed
FROM provider_json_demo.provider_network LIMIT 5;


### Scenario 17 — `variant_get` and `try_variant_get`

`variant_get` gives explicit control over the extraction path and return type.
`try_variant_get` is the safe version — returns NULL instead of error for missing paths.


In [ ]:
%sql
SELECT provider_id,
  variant_get(provider_data, '$.specialties[0].description', 'STRING') AS primary_specialty,
  variant_get(provider_data, '$.specialties[0].board_certified', 'BOOLEAN') AS is_board_certified,
  variant_get(provider_data, '$.contract.discount_pct', 'DOUBLE') AS discount_pct,
  variant_get(provider_data, '$.contract.fee_schedule_id', 'STRING') AS fee_schedule
FROM provider_json_demo.provider_network LIMIT 10;


In [ ]:
%sql
SELECT provider_id,
  try_variant_get(provider_data, '$.specialties[1].description', 'STRING') AS second_specialty,
  try_variant_get(provider_data, '$.name.organization_name', 'STRING') AS org_name,
  try_variant_get(provider_data, '$.nonexistent.path', 'STRING') AS missing_returns_null
FROM provider_json_demo.provider_network LIMIT 10;


### Scenario 18 — `variant_explode`: flatten VARIANT arrays

`variant_explode` is the VARIANT-native equivalent of `explode`.
It returns `(pos, key, value)` columns.  For arrays, `key` is NULL.


In [ ]:
%sql
-- Explode addresses array
SELECT p.provider_id, p.provider_type,
  addr.pos AS address_index,
  addr.value:type::string   AS address_type,
  addr.value:city::string   AS city,
  addr.value:zip::string    AS zip,
  addr.value:phone::string  AS phone
FROM provider_json_demo.provider_network p,
LATERAL variant_explode(p.provider_data:addresses) addr
LIMIT 15;


In [ ]:
%sql
-- Explode network participation — one row per provider-plan combination
SELECT p.provider_id,
  p.provider_data:name.last::string       AS provider_name,
  np.value:plan_code::string              AS plan_code,
  np.value:network_tier::string           AS network_tier,
  np.value:accepting_new_patients::boolean AS accepting_new
FROM provider_json_demo.provider_network p,
LATERAL variant_explode(p.provider_data:network_participation) np
WHERE p.provider_type = 'individual'
LIMIT 20;


### Scenario 19 — `schema_of_variant` and `schema_of_variant_agg`

`schema_of_variant_agg` is especially useful — it shows the union of all fields
across all rows, catching structural variations you would miss from a single row.


In [ ]:
%sql
SELECT provider_id, schema_of_variant(provider_data) AS row_schema
FROM provider_json_demo.provider_network LIMIT 3;


In [ ]:
%sql
-- Aggregated schema across all providers — shows the full possible structure
SELECT schema_of_variant_agg(provider_data) AS full_schema
FROM provider_json_demo.provider_network;


### Scenario 20 — Complex VARIANT: filtering + chained extraction

Realistic query: find all individual providers accepting new Medicare Advantage
patients in Tier 1, along with their specialty and city.


In [ ]:
%sql
SELECT
  p.provider_id, p.npi,
  p.provider_data:name.first::string       AS first_name,
  p.provider_data:name.last::string        AS last_name,
  p.provider_data:name.credentials::string AS credentials,
  variant_get(p.provider_data, '$.specialties[0].description', 'STRING') AS specialty,
  np.value:plan_code::string               AS plan_code,
  np.value:network_tier::string            AS tier,
  p.provider_data:addresses[0].city::string AS city,
  p.provider_data:contract.contract_type::string AS contract_type
FROM provider_json_demo.provider_network p,
LATERAL variant_explode(p.provider_data:network_participation) np
WHERE p.provider_type = 'individual'
  AND np.value:plan_code::string LIKE 'MA-%'
  AND np.value:accepting_new_patients::boolean = true
  AND np.value:network_tier::string = 'tier1'
ORDER BY p.provider_data:name.last::string;


In [ ]:
%sql
-- Languages spoken by specialty — network adequacy analysis
SELECT
  variant_get(p.provider_data, '$.specialties[0].description', 'STRING') AS specialty,
  lang.value::string AS language,
  count(*) AS provider_count
FROM provider_json_demo.provider_network p,
LATERAL variant_explode(p.provider_data:languages) lang
WHERE p.provider_type = 'individual'
GROUP BY 1, 2
ORDER BY specialty, provider_count DESC;


---
## Part 5 — Capstone: Cross-Table Join

This query joins all four tables, extracts from STRING JSON and VARIANT columns,
explodes a service-lines array, and produces a single flat analytics-ready dataset.

It demonstrates everything in this tutorial in one place:
- Colon extraction on STRING columns (claims, members, ops)
- Colon extraction on VARIANT columns (providers)
- `variant_get` with explicit typing
- `LATERAL VIEW explode` for service line arrays
- LEFT JOINs using JSON-extracted keys
- CTEs for readability


### Scenario 21 — Analytics-ready flattened dataset


In [ ]:
%sql
WITH
claim_lines AS (
  SELECT
    c.claim_id, c.submitted_date, c.claim_type, c.total_charge_amount,
    c.claim_json:subscriber.member_id::string     AS member_id,
    c.claim_json:filing_code::string              AS filing_code,
    c.claim_json:adjudication.status::string      AS adj_status,
    c.claim_json:adjudication.paid_amount::double  AS paid_amount,
    c.claim_json:adjudication.allowed_amount::double AS allowed_amount,
    c.claim_json:adjudication.denial_codes        AS denial_codes,
    c.claim_json:billing_provider.name::string    AS billing_provider_name,
    sl.line_number, sl.procedure_code,
    sl.charge_amount AS line_charge,
    sl.place_of_service, sl.rendering_provider_npi,
    sl.modifiers, sl.revenue_code
  FROM claims_json_demo.claims_submissions c
  LATERAL VIEW explode(
    from_json(c.claim_json:service_lines,
      'ARRAY<STRUCT<line_number:INT, procedure_code:STRING, modifiers:ARRAY<STRING>,
       charge_amount:DOUBLE, place_of_service:STRING,
       rendering_provider_npi:STRING, revenue_code:STRING>>')
  ) sl_tbl AS sl
),
member_details AS (
  SELECT
    m.member_id, m.line_of_business, m.plan_code,
    m.profile_json:demographics.first_name::string    AS first_name,
    m.profile_json:demographics.last_name::string     AS last_name,
    m.profile_json:demographics.address.city::string  AS city,
    m.profile_json:risk_scores.hcc_score::double      AS hcc_score,
    m.profile_json:risk_scores.sdoh_risk_level::string AS sdoh_risk,
    m.profile_json:sdoh_flags.food_insecurity::boolean AS food_insecurity,
    size(from_json(m.profile_json:conditions, 'ARRAY<STRUCT<code:STRING>>')) AS condition_count,
    size(from_json(m.profile_json:programs_enrolled, 'ARRAY<STRING>>'))      AS program_count
  FROM member_json_demo.member_profiles m
),
provider_details AS (
  SELECT
    p.npi, p.provider_type,
    COALESCE(p.provider_data:name.last::string,
             p.provider_data:name.organization_name::string) AS provider_name,
    variant_get(p.provider_data, '$.specialties[0].description', 'STRING') AS specialty,
    p.provider_data:contract.contract_type::string AS contract_type,
    np.value:plan_code::string   AS plan_code,
    np.value:network_tier::string AS network_tier
  FROM provider_json_demo.provider_network p,
  LATERAL variant_explode(p.provider_data:network_participation) np
),
ops_summary AS (
  SELECT
    related_claim_id AS claim_id,
    count(*) AS event_count,
    count_if(event_json:sla.met::boolean = false) AS sla_misses,
    max(event_json:status::string) AS latest_event_status,
    max(CASE WHEN event_type = 'prior_auth' THEN event_json:resolution.outcome::string END) AS pa_outcome
  FROM ops_json_demo.operational_events
  WHERE related_claim_id IS NOT NULL
  GROUP BY related_claim_id
)
SELECT
  cl.claim_id, cl.submitted_date, cl.claim_type,
  cl.procedure_code, cl.line_charge, cl.adj_status, cl.paid_amount,
  md.first_name, md.last_name, md.line_of_business,
  md.hcc_score, md.sdoh_risk, md.condition_count,
  pd.provider_name AS rendering_provider, pd.specialty, pd.network_tier,
  COALESCE(os.event_count, 0) AS related_events,
  COALESCE(os.sla_misses, 0) AS sla_misses
FROM claim_lines cl
LEFT JOIN member_details md ON cl.member_id = md.member_id
LEFT JOIN provider_details pd ON cl.rendering_provider_npi = pd.npi AND md.plan_code = pd.plan_code
LEFT JOIN ops_summary os ON cl.claim_id = os.claim_id
ORDER BY cl.submitted_date DESC, cl.claim_id
LIMIT 50;


### Bonus — Summary analytics from the joined dataset

Once the JSON is extracted and joined, standard SQL analytics work exactly as expected.


In [ ]:
%sql
-- Denial rate by SDoH risk level and line of business
WITH claim_data AS (
  SELECT c.claim_id,
    c.claim_json:subscriber.member_id::string AS member_id,
    c.claim_json:adjudication.status::string  AS adj_status
  FROM claims_json_demo.claims_submissions c
)
SELECT
  m.line_of_business,
  m.profile_json:risk_scores.sdoh_risk_level::string AS sdoh_risk,
  count(DISTINCT cd.claim_id) AS total_claims,
  count(DISTINCT CASE WHEN cd.adj_status = 'denied' THEN cd.claim_id END) AS denied_claims,
  round(count(DISTINCT CASE WHEN cd.adj_status = 'denied' THEN cd.claim_id END) * 100.0 /
    NULLIF(count(DISTINCT cd.claim_id), 0), 1) AS denial_rate_pct
FROM claim_data cd
JOIN member_json_demo.member_profiles m ON cd.member_id = m.member_id
GROUP BY 1, 2 ORDER BY 1, 2;


In [ ]:
%sql
-- SLA performance by event type — ops analytics
SELECT
  event_type,
  count(*) AS total_events,
  count_if(event_json:sla.met::boolean = true)  AS sla_met,
  count_if(event_json:sla.met::boolean = false) AS sla_missed,
  round(count_if(event_json:sla.met::boolean = true) * 100.0 / count(*), 1) AS sla_met_pct,
  round(avg(event_json:sla.actual_hours::double), 1) AS avg_actual_hours,
  round(percentile_approx(event_json:sla.actual_hours::double, 0.95), 1) AS p95_hours
FROM ops_json_demo.operational_events
GROUP BY event_type ORDER BY event_type;
